# Load EURUSD Data From Database

Scratch notebook for pulling raw daily candles from `fx_candles.db` and eyeballing
recent data before building indicators on top of it.

- Fetches every stored daily candle for `INSTRUMENT` in `YEAR`.
- Prints full detail for the most recent `LAST_N_DAYS` trading days.
- Re-run all cells to refresh against the latest data in `fx_candles.db`.

In [4]:
from __future__ import annotations

import sqlite3
from typing import Final

import pandas as pd

In [5]:
DB_PATH: Final[str] = "../../fx_candles.db"
TABLE: Final[str] = "candles_D"
INSTRUMENT: Final[str] = "EUR_USD"
YEAR: Final[int] = 2026
LAST_N_DAYS: Final[int] = 30

In [6]:
query = (
    f"SELECT * FROM {TABLE} "
    "WHERE instrument = ? AND time >= ? AND time < ? AND complete = 1 "
    "ORDER BY time ASC"
)
with sqlite3.connect(DB_PATH) as conn:
    candles = pd.read_sql_query(
        query, conn, params=(INSTRUMENT, f"{YEAR}-01-01", f"{YEAR + 1}-01-01")
    )

# print(candles.head())
print(f"Fetched {len(candles)} {INSTRUMENT} candles for {YEAR}.")

Fetched 136 EUR_USD candles for 2026.


In [7]:
candles["date"] = pd.to_datetime(candles["time"]).dt.date

# Get all unique dates in the candles DF -> sort them chronologically -> take the last N dates of them
last_30_dates = sorted(candles["date"].unique())[-LAST_N_DAYS:]
last_30_days = candles[candles["date"].isin(last_30_dates)]
# print(last_30_days)

print(f"Last {len(last_30_dates)} trading days: {last_30_dates[0]} → {last_30_dates[-1]}")

Last 30 trading days: 2026-05-31 → 2026-07-09


In [8]:
with pd.option_context("display.max_rows", None, "display.max_columns", None,
                        "display.width", None):
    print(last_30_days.to_string(index=False))

instrument                      time   bid_o   bid_h   bid_l   bid_c   ask_o   ask_h   ask_l   ask_c  volume  complete       date
   EUR_USD 2026-05-31T21:00:00+00:00 1.16521 1.16640 1.16060 1.16313 1.16565 1.16656 1.16076 1.16329  123672         1 2026-05-31
   EUR_USD 2026-06-01T21:00:00+00:00 1.16251 1.16550 1.16127 1.16300 1.16349 1.16567 1.16143 1.16319   84004         1 2026-06-01
   EUR_USD 2026-06-02T21:00:00+00:00 1.16249 1.16327 1.15941 1.15978 1.16349 1.16359 1.15957 1.15994  105799         1 2026-06-02
   EUR_USD 2026-06-03T21:00:00+00:00 1.15953 1.16448 1.15938 1.16102 1.16042 1.16462 1.15954 1.16119   90191         1 2026-06-03
   EUR_USD 2026-06-04T21:00:00+00:00 1.16116 1.16438 1.15172 1.15177 1.16192 1.16452 1.15187 1.15274  131400         1 2026-06-04
   EUR_USD 2026-06-07T21:00:00+00:00 1.15090 1.15540 1.14989 1.15314 1.15140 1.15557 1.15004 1.15333  144995         1 2026-06-07
   EUR_USD 2026-06-08T21:00:00+00:00 1.15347 1.15773 1.15262 1.15431 1.15392 1.15790 1.152